# 🤖 Sepsis CDSS: Model Training, Benchmarking & Evaluation Pipeline
**Project**: Web-Based Clinical Decision Support System (CDSS) for Early Sepsis Detection  
**Primary Algorithm**: Random Forest Classifier (100 Trees, `class_weight='balanced'`)  
**Benchmarking Suite**: Logistic Regression, Single Decision Tree, Random Forest Classifier  

---

### Pipeline Objectives:
1. **Clinical Feature Contract Enforcement**: Select the 15 Optimal Clinical Features (`FEATURE_NAMES`) required by the CDSS backend.
2. **Fallback Imputation Medians**: Calculate and serialize feature medians to `model/feature_medians.json`.
3. **Stratified Train/Test Split**: Maintain exact class balance across 80% train and 20% test partitions.
4. **Multi-Model Training**: Train Logistic Regression, Decision Tree, and Random Forest Classifier with balanced class weights.
5. **Comprehensive Clinical Evaluation**:
   - Classification Reports (Precision, Sensitivity/Recall, Specificity, F1-Score)
   - Receiver Operating Characteristic (ROC) & AUROC Comparison
   - Precision-Recall (PR) Curves & PR-AUC (AUPRC) Comparison
   - Confusion Matrices Heatmaps
6. **Feature Importance Analysis**: Global Gini importance ranking for clinical explainability.
7. **Production Model Export**: Save `sepsis_model.pkl` and verify startup contract compatibility with `services/ml_service.py`.

In [ ]:
import os
import sys
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve
)
import joblib

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline
print("Libraries imported successfully!")

## Step 1: Feature Contract Definition
The CDSS inference engine (`services/ml_service.py`) mandates the **15 Optimal Clinical Features**:
- **7 Vital Signs**: `HR`, `O2Sat`, `Temp`, `SBP`, `MAP`, `DBP`, `Resp`
- **5 Laboratory Biomarkers**: `WBC`, `Glucose`, `Lactate`, `Creatinine`, `Platelets`
- **3 Demographic & Stay Attributes**: `Age`, `Gender`, `ICULOS`

*Clinical Rationale*: PhysioNet features 16-40 suffer from $>90\%$ missingness in real-world ICUs. Training on 15 core biomarkers avoids synthetic imputation noise and delivers superior generalization.

In [ ]:
OPTIMAL_FEATURES = [
    "HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp",
    "WBC", "Glucose", "Lactate", "Creatinine", "Platelets",
    "Age", "Gender", "ICULOS"
]

print(f"15 Optimal Clinical Features ({len(OPTIMAL_FEATURES)}):")
for idx, feat in enumerate(OPTIMAL_FEATURES, start=1):
    print(f"  {idx:2d}. {feat}")

## Step 2: Dataset Loading & Imputation Medians
We load `physionet_sepsis_processed.csv` (or `physionet_sepsis_combined.csv`). We compute population medians for the 15 features and persist them to `model/feature_medians.json` for fallback imputation during live inference.

In [ ]:
csv_paths = [
    'physionet_sepsis_processed.csv',
    '../physionet_sepsis_processed.csv',
    'physionet_sepsis_combined.csv',
    '../physionet_sepsis_combined.csv'
]

dataset_path = None
for path in csv_paths:
    if os.path.exists(path):
        dataset_path = path
        break

if dataset_path is None:
    raise FileNotFoundError("Could not find sepsis dataset CSV file.")

print(f"Loading dataset from: '{dataset_path}'")
df = pd.read_csv(dataset_path)
print(f"Loaded {len(df):,} records with {len(df.columns)} columns.")

X = df[OPTIMAL_FEATURES].copy()
y = df['SepsisLabel'].values

# Compute and serialize feature medians
medians = X.median().to_dict()
os.makedirs('model', exist_ok=True)
with open('model/feature_medians.json', 'w') as f:
    json.dump(medians, f, indent=4)
print("Saved feature medians to 'model/feature_medians.json':")
for k, v in medians.items():
    print(f"  {k:12s}: {v:.2f}")

# Impute any remaining NaNs with column medians
X_imputed = X.fillna(medians)

## Step 3: Stratified Train / Test Split
To evaluate generalization on unseen data while preserving class proportions, we partition the dataset into 80% Training and 20% Testing subsets stratified by `y` (`SepsisLabel`).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Total Records:    {len(X_imputed):,}")
print(f"Training Subset:  {len(X_train):,} (Sepsis: {y_train.sum():,} | {y_train.mean()*100:.2f}%)")
print(f"Testing Subset:   {len(X_test):,} (Sepsis: {y_test.sum():,} | {y_test.mean()*100:.2f}%)")

## Step 4: Multi-Model Training & Benchmarking
We benchmark 3 models representing distinct algorithmic paradigms:
1. **Logistic Regression**: Scaled linear baseline (`class_weight='balanced'`).
2. **Decision Tree**: Fully interpretable hierarchical rule tree (`max_depth=8`, `class_weight='balanced'`).
3. **Random Forest Classifier**: Ensemble of 100 decorrelated trees (`max_depth=15`, `class_weight='balanced'`, `n_jobs=-1`).

In [ ]:
models = {}

# 1. Logistic Regression (requires feature standardization)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training 1/3: Logistic Regression...")
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
t0 = time.time()
lr.fit(X_train_scaled, y_train)
lr_time = time.time() - t0
models['Logistic Regression'] = {'model': lr, 'is_scaled': True, 'train_time': lr_time}

# 2. Single Decision Tree
print("Training 2/3: Decision Tree Classifier...")
dt = DecisionTreeClassifier(max_depth=8, class_weight='balanced', random_state=42)
t0 = time.time()
dt.fit(X_train, y_train)
dt_time = time.time() - t0
models['Decision Tree'] = {'model': dt, 'is_scaled': False, 'train_time': dt_time}

# 3. Random Forest Classifier
print("Training 3/3: Random Forest Classifier (100 Trees)...")
rf = RandomForestClassifier(n_estimators=100, max_depth=15, class_weight='balanced', random_state=42, n_jobs=-1)
t0 = time.time()
rf.fit(X_train, y_train)
rf_time = time.time() - t0
models['Random Forest'] = {'model': rf, 'is_scaled': False, 'train_time': rf_time}

print("All models trained successfully!")

## Step 5: Comprehensive Clinical Performance Evaluation
We evaluate clinical discrimination across all standard diagnostic metrics:
- **Sensitivity / Recall**: Percentage of actual sepsis cases correctly flagged.
- **Specificity**: Percentage of non-septic patients correctly identified (preventing alarm fatigue).
- **PR-AUC (Precision-Recall Area Under Curve)**: The gold standard metric for severe class imbalance.
- **ROC-AUC (Receiver Operating Characteristic Area Under Curve)**: Overall discriminatory power.

In [ ]:
results = []
predictions_dict = {}
probabilities_dict = {}

for name, item in models.items():
    clf = item['model']
    X_eval = X_test_scaled if item['is_scaled'] else X_test
    
    preds = clf.predict(X_eval)
    probs = clf.predict_proba(X_eval)[:, 1]
    
    predictions_dict[name] = preds
    probabilities_dict[name] = probs
    
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds, zero_division=0),
        'Recall / Sensitivity': recall_score(y_test, preds),
        'Specificity': tn / (tn + fp),
        'F1-Score': f1_score(y_test, preds),
        'ROC-AUC': roc_auc_score(y_test, probs),
        'PR-AUC': average_precision_score(y_test, probs),
        'Train Time (s)': item['train_time']
    })

comparison_df = pd.DataFrame(results).set_index('Model')
print("--- CLINICAL PERFORMANCE COMPARISON TABLE ---")
display_df = comparison_df.round(4)
display_df

## Step 6: Diagnostic Visualizations: ROC & PR Curves
- **ROC Curves**: Display True Positive Rate vs False Positive Rate.
- **Precision-Recall Curves**: Critical for medical alerting where False Positives induce clinician alarm fatigue.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = {'Logistic Regression': '#e67e22', 'Decision Tree': '#2ecc71', 'Random Forest': '#2980b9'}

# 1. ROC Curves
for name, probs in probabilities_dict.items():
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc_val = roc_auc_score(y_test, probs)
    axes[0].plot(fpr, tpr, lw=2.2, color=colors[name], label=f"{name} (AUROC = {auc_val:.3f})")

axes[0].plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Guess (AUC = 0.500)')
axes[0].set_title('Receiver Operating Characteristic (ROC) Comparison', fontsize=13, fontweight='bold')
axes[0].set_xlabel('False Positive Rate (1 - Specificity)', fontsize=11)
axes[0].set_ylabel('True Positive Rate (Sensitivity / Recall)', fontsize=11)
axes[0].legend(loc='lower right', frameon=True)

# 2. Precision-Recall Curves
baseline_prec = y_test.mean()
for name, probs in probabilities_dict.items():
    prec, rec, _ = precision_recall_curve(y_test, probs)
    pr_auc_val = average_precision_score(y_test, probs)
    axes[1].plot(rec, prec, lw=2.2, color=colors[name], label=f"{name} (PR-AUC = {pr_auc_val:.3f})")

axes[1].axhline(baseline_prec, color='black', linestyle='--', label=f'Baseline Rate ({baseline_prec:.3f})')
axes[1].set_title('Precision-Recall (PR) Curve Comparison', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Recall (Sensitivity)', fontsize=11)
axes[1].set_ylabel('Precision (Positive Predictive Value)', fontsize=11)
axes[1].legend(loc='upper right', frameon=True)

plt.tight_layout()
plt.show()

## Step 7: Confusion Matrices
Detailed breakdown of True Negatives, False Positives (over-alerting), False Negatives (missed sepsis), and True Positives.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, preds) in zip(axes, predictions_dict.items()):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=['Pred Non-Sepsis', 'Pred Sepsis'],
                yticklabels=['Actual Non-Sepsis', 'Actual Sepsis'])
    ax.set_title(f'{name} Confusion Matrix', fontsize=12, fontweight='bold')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

## Step 8: Random Forest Feature Importance
Gini Impurity (Mean Decrease in Impurity) reveals the relative contribution of each clinical biomarker to the ensemble's sepsis predictions.

In [ ]:
importances = rf.feature_importances_
feat_imp_df = pd.DataFrame({
    'Feature': OPTIMAL_FEATURES,
    'Importance': importances
}).sort_values(by='Importance', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(feat_imp_df['Feature'], feat_imp_df['Importance'], color='#2b5c8f', edgecolor='navy')
plt.title('Random Forest Feature Importance (15 Optimal Clinical Biomarkers)', fontsize=13, fontweight='bold')
plt.xlabel('Gini Importance (MDI)', fontsize=11)
plt.ylabel('Clinical Features', fontsize=11)
plt.tight_layout()
plt.show()

print("Top Clinical Predictors of Sepsis Risk:")
for _, row in feat_imp_df.tail(6).iloc[::-1].iterrows():
    print(f"  - {row['Feature']:12s}: {row['Importance']:.4f}")

## Step 9: Production Model Export & Startup Verification
We serialize the trained Random Forest model to `sepsis_model.pkl`.  
We then execute the **Startup Contract Verification** to guarantee exact compatibility with `services/ml_service.py` in the Flask CDSS web app.

In [ ]:
# Save model artifact
output_model_path = 'sepsis_model.pkl'
joblib.dump(rf, output_model_path)
print(f"Saved production model to '{output_model_path}' ({os.path.getsize(output_model_path)/(1024*1024):.1f} MB).")

# Verification: Load back and test feature contract
loaded_model = joblib.load(output_model_path)
loaded_features = list(getattr(loaded_model, "feature_names_in_", []))

print(f"Loaded model features ({len(loaded_features)}): {loaded_features}")
if loaded_features == OPTIMAL_FEATURES:
    print("SUCCESS: Model feature names exactly match OPTIMAL_FEATURES contract!")
else:
    print("WARNING: Feature names mismatch!")

# Simulate test inference with population median vitals
test_sample = pd.DataFrame([medians])
prob = loaded_model.predict_proba(test_sample)[0, 1]
print(f"Test inference with median patient vector -> Sepsis Probability: {prob:.4f}")
print("Sepsis CDSS model is fully trained, verified, and ready for deployment!")